# Ch02 练习参考答案：Bridge Grid 中 γ 如何翻转策略

> 我们找 γ* 使得最优策略从"绕远"切到"抄近道"。

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from utils import set_seed
from rlenvs import bridge_grid

set_seed(0)


def value_iteration(env, gamma, theta=1e-8, max_iters=10000):
    V = np.zeros(env.nS)
    for it in range(max_iters):
        Q = env.R + gamma * np.einsum('saq,q->sa', env.P, V)
        new_V = Q.max(axis=1)
        for s in range(env.nS):
            if env.is_terminal(s):
                new_V[s] = 0.0
        delta = np.abs(new_V - V).max()
        V = new_V
        if delta < theta:
            break
    return V


env = bridge_grid(seed=0)
print(f"shape: {env.shape}, 终点: {env.terminals}")
# 终点 (1, 4) = 1*5+4 = 9
# 桥起点 (1, 0) = 1*5+0 = 5
# 绕远起点 (0, 0) = 0

# 扫描 γ 找 V*(bridge_start) vs V*(roundabout_start)
gammas = np.linspace(0.0, 0.99, 100)
v_bridge = []
v_round = []
for g in gammas:
    V = value_iteration(env, gamma=g)
    v_bridge.append(V[5])   # 桥起点
    v_round.append(V[0])    # 绕远起点

v_bridge = np.array(v_bridge)
v_round = np.array(v_round)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(gammas, v_bridge, label='V*(1,0) bridge 起点（直接走桥）', linewidth=2)
ax.plot(gammas, v_round, label='V*(0,0) 绕远起点', linewidth=2)
ax.set_xlabel('γ')
ax.set_ylabel('V*')
ax.set_title('Bridge Grid：γ 越大，绕远越值')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 找最优动作在每个 γ 下从 (1, 0) 出发是什么
print(f"\n从起点 (1, 0) 在不同 γ 下的最优策略：")
for g in [0.0, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]:
    V = value_iteration(env, gamma=g)
    s = 5  # (1, 0)
    # 计算 Q[s, a]
    Q = env.R + g * np.einsum('saq,q->sa', env.P, V)
    best_a = int(np.argmax(Q[s]))
    names = ['↑', '→', '↓', '←']
    print(f"  γ={g:.2f}: 最优动作 = {names[best_a]}, V*(1,0) = {V[s]:.3f}")